# Stage 1 — Deflationary control: Flat vs Flat + LLRD (slow-deep and fast-deep)

The comparison that decides the project (plan §1.2). Both arms use the same network, data order (seed),
batch size and epochs, so parameter count and optimiser-step count are identical. The summary cell **asserts** this.
`flat_llrd` (slow-deep): shallow→deep at lr/1, lr/4, lr/16. `flat_llrd_rev` (fast-deep): shallow→deep at lr/16, lr/4, lr/1.
The head is in the fast level in both. An arm whose `LR` is still `None` is skipped, so you can run the others first.
The earlier Flat runs at lr 0.3 stay on Drive as a record; this notebook runs Flat at the selected lr 0.1.
Before running: settle the head placement and "matched effective lr" with your supervisor,
and fill `LR` from `stage1_lr_grid`. Checkpoints (~90 MB/run) go to `MyDrive/fyp_nested_cl/checkpoints/`.
Nesting arms A–D get added here after you approve the next step.

In [ ]:
# ---- Setup: Drive + code + wandb (run this first, every session) ----
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"   # <-- EDIT ONCE
BRANCH = "main"

import os, subprocess, sys
from google.colab import drive, userdata

# Drive must be mounted from the notebook kernel (never from a !python subprocess).
drive.mount("/content/drive")

# Code: fresh clone each session (Colab disk is wiped on reset), pull if it already exists.
# For a private repo, add a Colab secret GITHUB_TOKEN (a fine-grained read-only token).
CODE = "/content/fyp"
url = REPO_URL
try:
    tok = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if os.path.isdir(CODE):
    subprocess.run(["git", "-C", CODE, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, CODE], check=True)
print(subprocess.check_output(["git", "-C", CODE, "log", "-1", "--oneline"]).decode())

# Dependencies (torch/torchvision come preinstalled with Colab's CUDA build; don't reinstall them).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)

# wandb: key comes from the Colab secret WANDB_API_KEY (key icon in the left sidebar). Never hardcode it.
import wandb
wandb.login(key=userdata.get("WANDB_API_KEY"))

sys.path.insert(0, f"{CODE}/src")
# Python caches imported modules for the whole kernel session, so after a git pull
# a plain `import` would silently keep running the OLD code. Drop the cached copies.
for _m in ["fyp_infra", "stage1_runner", "stage1_analysis", "stage1_drift_diagnostic"]:
    sys.modules.pop(_m, None)
import torch
from fyp_infra import setup_paths, run_sweep, run_id, load_result
import stage1_runner as sr
paths = setup_paths()   # -> /content/drive/MyDrive/fyp_nested_cl/{data,results,logs}
print(paths)
# Hard stop for training: a CPU run is ~12x slower and its timings are useless for budgeting.
REQUIRE_GPU = True
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime)")

In [ ]:
EPOCHS = 10            # from the epoch sweep
SEEDS = [0, 1, 2]
# Base (fast-level) lr per arm, from stage1_lr_grid. Fill these in before running.
LR = {"flat": 0.1, "flat_llrd": 0.5, "flat_llrd_rev": None}   # rev: fill in from stage1_lr_grid
ARMS = {
    "flat":          dict(lr_schedule="flat"),
    "flat_llrd":     dict(lr_schedule="llrd", level_ratios=(1, 4, 16), level_order="slow_deep"),
    "flat_llrd_rev": dict(lr_schedule="llrd", level_ratios=(1, 4, 16), level_order="fast_deep"),
}
for name in ARMS:
    if LR[name] is None: print(f"skipping {name}: set LR['{name}'] from stage1_lr_grid first")
CONFIGS = [sr.make_cfg(stage="stage1_control", strategy="naive", epochs=EPOCHS, seed=s, lr=LR[name],
                       eval_split="test",          # final comparison -> real test set
                       save_checkpoints=True,      # weights after every task, for CKA / probes later
                       **arm)
           for s in SEEDS for name, arm in ARMS.items() if LR[name] is not None]
SHOW = ["lr", "seed", "epochs"]
for c in CONFIGS: print(run_id(c), sr.arm_label(c), {k: c[k] for k in SHOW})

In [ ]:
out = run_sweep(CONFIGS, sr.run_experiment, paths)

In [ ]:
import numpy as np, pandas as pd
rows = []
for cfg in CONFIGS:
    r = load_result(cfg, paths)
    if r is None:
        rows.append({**{k: cfg[k] for k in SHOW}, "status": "not finished"}); continue
    m = r["metrics"]
    rows.append({"arm": sr.arm_label(cfg), **{k: cfg[k] for k in SHOW}, "run_id": r["run_id"],
                 "final_avg_acc": m["final_avg_acc"], "avg_forgetting": m["avg_forgetting"],
                 "avg_incremental_acc": m["avg_incremental_acc"],
                 # mean of the diagonal: how well each task was learned when it was current.
                 # Report next to forgetting: forgetting = peak - final, so it grows with this.
                 "acc_just_learned": float(np.mean(np.diag(m["acc_matrix"]))),
                 "steps": m["optimizer_steps"], "n_params": m["n_params"], "minutes": r["wall_clock_s"] / 60})
df = pd.DataFrame(rows); df

In [ ]:
done = df[df.get("status") != "not finished"] if "status" in df else df
assert done["steps"].nunique() == 1 and done["n_params"].nunique() == 1, "steps or param count differ between arms. The comparison is NOT matched"
print(done.groupby(["arm", "lr"])[["final_avg_acc", "avg_forgetting", "avg_incremental_acc"]].agg(["mean", "std", "count"]))
print(done.groupby(["arm", "lr"])["acc_just_learned"].agg(["mean", "std"]))